In [1]:
import pandas as pd
import torch

_GROUPS = ["women", "black", "asian", "muslim", "mental_dis"]
_N_PER_CLASS = 230              # 250 toxic + 250 non-toxic per group.
_HUMAN_TOXIC_THRESHOLD = 3.0    # On the 1-5 toxicity_human scale.
_SEED = 42
_MODEL_NAME = "facebook/roberta-hate-speech-dynabench-r4-target"

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [ ]:
toxigen_df = pd.read_csv("annotated_train.csv")
print(toxigen_df.shape)  # Expect (8960, ...)

(8960, 14)


In [ ]:
toxigen_df["true_label"] = (
    toxigen_df["toxicity_human"] >= _HUMAN_TOXIC_THRESHOLD
).astype(int)
filtered = toxigen_df[toxigen_df["target_group"].isin(_GROUPS)]
filtered = filtered.drop_duplicates(subset="text")

counts = filtered.groupby(["target_group", "true_label"]).size().unstack()
counts.columns = ["non_toxic", "toxic"]
print(counts)

              non_toxic  toxic
target_group                  
asian               471    231
black               420    293
mental_dis          472    242
muslim              437    251
women               446    271


In [ ]:
too_small = counts[(counts < _N_PER_CLASS).any(axis=1)]
assert too_small.empty, (
    f"Not enough statements for {_N_PER_CLASS} per class:\n{too_small}")

data_df = (filtered
           .groupby(["target_group", "true_label"], group_keys=False)
           .sample(n=_N_PER_CLASS, random_state=_SEED)
           .sample(frac=1, random_state=_SEED)
           .reset_index(drop=True))

print(len(data_df))  # Expect 2500.
print(data_df.groupby(["target_group", "true_label"]).size().unstack())

2300
true_label      0    1
target_group          
asian         230  230
black         230  230
mental_dis    230  230
muslim        230  230
women         230  230


In [ ]:
from google.colab import files

data_df.to_csv("toxigen_processed.csv", index=False)
files.download("toxigen_processed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
data_df = pd.read_csv("toxigen_processed.csv")
print(data_df.shape)

(2300, 15)


In [3]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=_MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
    truncation=True,
)
results = classifier(data_df["text"].astype(str).tolist(), batch_size=64)

data_df["hate_prob"] = [
    next(s["score"] for s in r if s["label"] == "hate") for r in results
]
data_df["pred_label"] = (data_df["hate_prob"] >= 0.5).astype(int)

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [4]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score)

y_true = data_df["true_label"]
y_pred = data_df["pred_label"]

print(f"Accuracy: {(y_true == y_pred).mean():.3f}")
print(classification_report(y_true, y_pred,
                            target_names=["not toxic", "toxic"], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_true, data_df['hate_prob']):.3f}")

cm_norm = confusion_matrix(y_true, y_pred, normalize="true")
cell_names = [["TN", "FP"], ["FN", "TP"]]
cm_df = pd.DataFrame(
    [[f"{cell_names[i][j]}: {cm_norm[i, j]:.3f}" for j in range(2)]
     for i in range(2)],
    index=["true: not toxic", "true: toxic"],
    columns=["pred: not toxic", "pred: toxic"],
)
print("Confusion matrix (normalized per true class):")
print(cm_df)

Accuracy: 0.677
              precision    recall  f1-score   support

   not toxic      0.621     0.908     0.738      1150
       toxic      0.829     0.447     0.581      1150

    accuracy                          0.677      2300
   macro avg      0.725     0.677     0.659      2300
weighted avg      0.725     0.677     0.659      2300

ROC-AUC: 0.806
Confusion matrix (normalized per true class):
                pred: not toxic pred: toxic
true: not toxic       TN: 0.908   FP: 0.092
true: toxic           FN: 0.553   TP: 0.447


In [5]:
def group_metrics(df: pd.DataFrame) -> pd.Series:

  negatives = df[df["true_label"] == 0]
  positives = df[df["true_label"] == 1]
  return pd.Series({
      "n": len(df),
      "accuracy": (df["true_label"] == df["pred_label"]).mean(),
      "fpr": negatives["pred_label"].mean(),
      "fnr": 1 - positives["pred_label"].mean(),
  })


per_group = (data_df.groupby("target_group")
             .apply(group_metrics)
             .sort_values("fnr", ascending=False))
per_group.round(3)

/tmp/ipykernel_885/2265389340.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(group_metrics)


,n,accuracy,fpr,fnr
target_group,,,,
mental_dis,460.0,0.611,0.078,0.700
black,460.0,0.678,0.091,0.552
muslim,460.0,0.667,0.143,0.522
women,460.0,0.713,0.065,0.509
asian,460.0,0.717,0.083,0.483


In [6]:
from google.colab import files

data_df.to_csv("toxigen_processed_scored.csv", index=False)
files.download("toxigen_processed_scored.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>